In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
paper_4.2_MDS_index_profiles_summary_v2.py
------------------------------------------
Purpose:
    Summarize MDS index profiles after R-side computation.
    Input  : allData/**/csv_profiles/index_profiles_all_*.csv
    Output : for_checking_20251127/4.2_clustering/results_summary/**

Directory Structure:

Input:
    <BASE_ROOT>/20251026_under_25clusters_program_and_result/for_checking_20251127/
        4.2_clustering/allData/
            classical/U10/OH/csv_profiles/*.csv
            classical/U10/FP/csv_profiles/*.csv
            nonmetric/U25/OH/csv_profiles/*.csv
            ...

Output:
    <BASE_ROOT>/20251026_under_25clusters_program_and_result/for_checking_20251127/
        4.2_clustering/results_summary/
            classical/U10/summaries/*.csv
            classical/U10/figures/profiles/*.png|pdf
            classical/U10/figures/kbest/*.png|pdf
            nonmetric/U25/...
"""

import os
import glob
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# ============================================================================
# CONFIG (edit here only)
# ============================================================================

BASE_ROOT = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No"

# NEW INPUT (your request): allData
INPUT_ROOT = os.path.join(
    BASE_ROOT,
    "20251026_under_25clusters_program_and_result",
    "for_checking_20251127",
    "4.2_clustering",
    "allData"
)

# NEW OUTPUT (your request)
OUTPUT_ROOT = os.path.join(
    BASE_ROOT,
    "20251026_under_25clusters_program_and_result",
    "for_checking_20251127",
    "4.2_clustering",
    "results_summary"
)

METHOD_TAGS = ["classical", "nonmetric"]
U_TAGS = ["U10", "U25", "U50"]
DATASETS = ["OH", "FP"]
MODES = ["top3", "cumeig"]

INDEX_NAMES = ["silhouette", "dunn", "gap", "ch", "db", "ptbiserial"]
TS = datetime.now().strftime("%Y%m%d_%H%M%S")


# ============================================================================
# Helpers
# ============================================================================

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)
    return p


def find_profiles(method_tag, U_tag, dataset, mode):
    """
    Find latest index_profiles_all CSV under:
    allData/{method_tag}/{U_tag}/{dataset}/csv_profiles/
    """
    d = os.path.join(INPUT_ROOT, method_tag, U_tag, dataset, "csv_profiles")
    pattern = f"index_profiles_all_{dataset}_{mode}_{method_tag}_{U_tag}_kmax*.csv"
    files = glob.glob(os.path.join(d, pattern))
    if not files:
        print(f"[WARN] No profile CSV: {d}/{pattern}")
        return None
    return max(files, key=os.path.getmtime)


def load_long_df(method_tag, U_tag, dataset, mode):
    f = find_profiles(method_tag, U_tag, dataset, mode)
    if f is None:
        return None
    df = pd.read_csv(f)

    for c in INDEX_NAMES:
        if c not in df.columns:
            df[c] = np.nan

    df_long = df.melt(id_vars=["k"], value_vars=INDEX_NAMES,
                      var_name="index", value_name="score")
    df_long["method"] = method_tag
    df_long["U_tag"] = U_tag
    df_long["dataset"] = dataset
    df_long["mode"] = mode
    return df_long


def plot_profiles(df_long, out_png, out_pdf, title):
    sns.set(style="whitegrid")
    df_long["index"] = pd.Categorical(df_long["index"], INDEX_NAMES, ordered=True)

    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    axes = axes.ravel()
    for i, idx in enumerate(INDEX_NAMES):
        ax = axes[i]
        tmp = df_long[df_long["index"] == idx].sort_values("k")
        if tmp.empty:
            ax.text(0.5, 0.5, "No Data", ha="center", va="center")
        else:
            ax.plot(tmp["k"], tmp["score"], marker="o")
        ax.set_title(idx)
        ax.set_xlabel("k")
        ax.set_ylabel("Score")

    fig.suptitle(title, fontsize=14, fontweight="bold")
    fig.tight_layout(rect=[0, 0.03, 1, 0.95])

    fig.savefig(out_png, dpi=300)
    fig.savefig(out_pdf)
    plt.close(fig)
    print("[Saved]", out_png)


def compute_best(df_long):
    rows = []
    for idx, g in df_long.groupby("index"):
        g = g.dropna(subset=["score"])
        if g.empty:
            rows.append({"index": idx, "k_best": np.nan, "score_best": np.nan})
            continue

        if idx.lower() == "db":
            row = g.loc[g["score"].idxmin()]
        else:
            row = g.loc[g["score"].idxmax()]

        rows.append({"index": idx,
                     "k_best": int(row["k"]),
                     "score_best": float(row["score"])})
    return pd.DataFrame(rows)


def plot_kbest_heatmap(df_kbest, out_png, out_pdf, title):
    """
    df_kbest: columns = index, dataset, mode, k_best
    """
    df = df_kbest.copy()
    df["dataset_mode"] = df["dataset"] + "_" + df["mode"]

    pivot = df.pivot_table(index="index", columns="dataset_mode",
                           values="k_best").reindex(INDEX_NAMES)

    sns.set(style="white")
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(pivot, annot=True, cmap="viridis", fmt=".0f",
                cbar_kws={"label": "Best k"}, ax=ax)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Dataset & Mode")
    ax.set_ylabel("Index")

    fig.tight_layout()
    fig.savefig(out_png, dpi=300)
    fig.savefig(out_pdf)
    plt.close(fig)
    print("[Saved]", out_png)


def plot_kbest_bar(df_kbest, out_png, out_pdf, title):
    sns.set(style="whitegrid")
    df = df_kbest.copy()
    df["index"] = pd.Categorical(df["index"], INDEX_NAMES, ordered=True)

    g = sns.FacetGrid(df, row="dataset", col="mode",
                      margin_titles=True, sharey=True,
                      height=3, aspect=1.4)
    g.map_dataframe(sns.barplot, x="index", y="k_best",
                    order=INDEX_NAMES, edgecolor="black")
    for ax in g.axes.flatten():
        for label in ax.get_xticklabels():
            label.set_rotation(25)
            label.set_ha("right")

    plt.subplots_adjust(top=0.85)
    g.fig.suptitle(title, fontsize=14, fontweight="bold")

    g.fig.savefig(out_png, dpi=300)
    g.fig.savefig(out_pdf)
    plt.close(g.fig)
    print("[Saved]", out_png)


# ============================================================================
# Main
# ============================================================================

def main():

    print("=== Python Summary (v2) ===")
    print("Input :", INPUT_ROOT)
    print("Output:", OUTPUT_ROOT)

    for method in METHOD_TAGS:
        method_out = ensure_dir(os.path.join(OUTPUT_ROOT, method))

        for U_tag in U_TAGS:
            print(f"\n=== {method} / {U_tag} ===")
            U_out = ensure_dir(os.path.join(method_out, U_tag))
            summary_dir = ensure_dir(os.path.join(U_out, "summaries"))
            fig_root = ensure_dir(os.path.join(U_out, "figures"))
            fig_profiles = ensure_dir(os.path.join(fig_root, "profiles"))
            fig_kbest = ensure_dir(os.path.join(fig_root, "kbest"))

            # Collect best ks
            best_rows = []

            # Generate profiles for each dataset × mode
            for dataset in DATASETS:
                for mode in MODES:
                    df_long = load_long_df(method, U_tag, dataset, mode)
                    if df_long is None:
                        continue

                    title = f"Index profiles — {method} — {U_tag} — {dataset} — {mode}"
                    base = f"profiles_{method}_{U_tag}_{dataset}_{mode}_{TS}"
                    out_png = os.path.join(fig_profiles, base + ".png")
                    out_pdf = os.path.join(fig_profiles, base + ".pdf")

                    plot_profiles(df_long, out_png, out_pdf, title)

                    # best k
                    df_k = compute_best(df_long)
                    df_k["method"] = method
                    df_k["U_tag"] = U_tag
                    df_k["dataset"] = dataset
                    df_k["mode"] = mode
                    best_rows.append(df_k)

            if not best_rows:
                print(f"[WARN] No best_k rows for {method}/{U_tag}. Skip.")
                continue

            df_best = pd.concat(best_rows, ignore_index=True)

            # Save long-format
            long_path = os.path.join(
                summary_dir,
                f"{method}_{U_tag}_kbest_long_{TS}.csv"
            )
            df_best.to_csv(long_path, index=False)
            print("[Saved]", long_path)

            # Save wide-format
            df_best["dataset_mode"] = df_best["dataset"] + "_" + df_best["mode"]
            df_wide = (
                df_best.pivot_table(
                    index="index", columns="dataset_mode", values="k_best"
                ).reindex(INDEX_NAMES)
            )
            wide_path = os.path.join(
                summary_dir,
                f"{method}_{U_tag}_kbest_wide_{TS}.csv"
            )
            df_wide.to_csv(wide_path)
            print("[Saved]", wide_path)

            # Heatmap
            heat_png = os.path.join(fig_kbest,
                                    f"kbest_heatmap_{method}_{U_tag}_{TS}.png")
            heat_pdf = os.path.join(fig_kbest,
                                    f"kbest_heatmap_{method}_{U_tag}_{TS}.pdf")
            plot_kbest_heatmap(
                df_best,
                heat_png,
                heat_pdf,
                f"Best k — {method} — {U_tag}"
            )

            # Bar plot
            bar_png = os.path.join(fig_kbest,
                                   f"kbest_bar_{method}_{U_tag}_{TS}.png")
            bar_pdf = os.path.join(fig_kbest,
                                   f"kbest_bar_{method}_{U_tag}_{TS}.pdf")
            plot_kbest_bar(
                df_best,
                bar_png,
                bar_pdf,
                f"Best k (dataset×mode) — {method} — {U_tag}"
            )

    print("\n✅ All done.")
    print("Results saved to:", OUTPUT_ROOT)


if __name__ == "__main__":
    main()


=== Python Summary (v2) ===
Input : /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/4.2_clustering/allData
Output: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/4.2_clustering/results_summary

=== classical / U10 ===
[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/4.2_clustering/results_summary/classical/U10/figures/profiles/profiles_classical_U10_OH_top3_20251128_110031.png


/tmp/ipykernel_73040/3427782674.py:145: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for idx, g in df_long.groupby("index"):


[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/4.2_clustering/results_summary/classical/U10/figures/profiles/profiles_classical_U10_OH_cumeig_20251128_110031.png


/tmp/ipykernel_73040/3427782674.py:145: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for idx, g in df_long.groupby("index"):


[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/4.2_clustering/results_summary/classical/U10/figures/profiles/profiles_classical_U10_FP_top3_20251128_110031.png


...

[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/4.2_clustering/results_summary/nonmetric/U50/figures/profiles/profiles_nonmetric_U50_FP_top3_20251128_110031.png


/tmp/ipykernel_73040/3427782674.py:145: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for idx, g in df_long.groupby("index"):


[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/4.2_clustering/results_summary/nonmetric/U50/figures/profiles/profiles_nonmetric_U50_FP_cumeig_20251128_110031.png
[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/4.2_clustering/results_summary/nonmetric/U50/summaries/nonmetric_U50_kbest_long_20251128_110031.csv


/tmp/ipykernel_73040/3427782674.py:145: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  for idx, g in df_long.groupby("index"):


[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/4.2_clustering/results_summary/nonmetric/U50/summaries/nonmetric_U50_kbest_wide_20251128_110031.csv
[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/4.2_clustering/results_summary/nonmetric/U50/figures/kbest/kbest_heatmap_nonmetric_U50_20251128_110031.png
[Saved] /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/4.2_clustering/results_summary/nonmetric/U50/figures/kbest/kbest_bar_nonmetric_U50_20251128_110031.png

✅ All done.
Results saved to: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/4.2_clustering/results_summary
